<a href="https://colab.research.google.com/github/IrisCheon/NLP-practice/blob/main/Customer_Support_Discourse_Analysis_on_Twitter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Support Tweet Topic Modeling

This project explores topic modeling on customer support tweets using TF-IDF and NMF. The initial goal was to identify semantic issue-related topics from customer complaints, while also examining how traditional bag-of-words topic modeling behaves on short and noisy conversational text.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")

print("Path to dataset files:", path)

100%|██████████| 169M/169M [00:01<00:00, 173MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/thoughtvector/customer-support-on-twitter/versions/10


In [4]:
print(os.listdir(path))

['sample.csv', 'twcs']


In [19]:
!file {path}/twcs

/root/.cache/kagglehub/datasets/thoughtvector/customer-support-on-twitter/versions/10/twcs: directory


#■ 데이터 확인

In [20]:
df = pd.read_csv(f"{path}/twcs/twcs.csv")
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB


In [23]:
df.columns

Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

In [24]:
df.isnull().sum()

,0
tweet_id,0
author_id,0
inbound,0
created_at,0
text,0
response_tweet_id,1040629
in_response_to_tweet_id,794335


In [25]:
customers = df[df["inbound"] == True].copy()
brands = df[df["inbound"] == False].copy()

In [26]:
customer_sample = customers.sample(30000, random_state = 42)
brand_sample = brands.sample(30000, random_state = 42)

In [27]:
import re

In [28]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [29]:
customer_sample["clean_text"] = customer_sample["text"].apply(clean_tweet)

In [31]:
customer_sample["word_count"] = customer_sample["clean_text"].str.split().str.len()
customer_sample = customer_sample[customer_sample["word_count"]>=5].copy()

customer_sample.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,clean_text,word_count
26861,34581,123491,True,Wed Nov 01 14:03:55 +0000 2017,@AppleSupport Basically for a chat to be opene...,34582,34579.0,basically for a chat to be opened from call lo...,23
211386,246537,174558,True,Thu Oct 05 14:08:30 +0000 2017,@AppleSupport iOS 11.02 and Watchos4.0: No ico...,"246536,246538",NaN,ios and watchos no icon for twitter notificati...,21
1225222,1351215,435088,True,Mon Oct 16 13:33:22 +0000 2017,@ATVIAssist Hi there! If I buy Call of Duty WW...,1351214,NaN,hi there if i buy call of duty wwii on steam t...,22
194583,228814,170570,True,Thu Oct 05 08:01:11 +0000 2017,Hi @Safaricom_Care why can't I pay my my Dstv ...,228812,NaN,hi why can t i pay my my dstv texts says the o...,15
1321593,1456209,457998,True,Thu Nov 02 18:45:26 +0000 2017,trying to buy my kid nibling a keyboard for an...,1456208,NaN,trying to buy my kid nibling a keyboard for an...,23


#■ TF-IDF + NMF

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [33]:
vectorizer = TfidfVectorizer(
    stop_words = "english",
    max_df = 0.8,
    min_df = 10,
    max_features = 5000,
    ngram_range = (1,2)
)

X = vectorizer.fit_transform(customer_sample["clean_text"])

In [35]:
n_topics = 6

nmf = NMF(
    n_components = n_topics,
    random_state = 42,
    max_iter = 300,
    init = "nndsvda"
)

W = nmf.fit_transform(X)    # 각 문서가 어떤 토픽을 갖는지?
H = nmf.components_     # 각 토픽이 어떤 단어를 중요하게 갖는지?

In [36]:
feature_names = vectorizer.get_feature_names_out()

def display_topics(model, feature_names, n_top_words = 10):
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_indices]

        print(f"\nTopic {topic_idx}")
        print(", ".join(top_words))

display_topics(nmf, feature_names, 15)


Topic 0
time, phone, don, ve, amp, order, like, app, fix, new, going, flight, day, update, today

Topic 1
service, customer, customer service, worst, worst customer, terrible, care, great, poor, customer care, today, bad, experience, horrible, good

Topic 2
thanks, hi, dm, reply, ok, great, ll, thanks help, thanks reply, ok thanks, response, check, sure, good, flight

Topic 3
help, need, need help, account, dm, hi, email, hey, trying, thanks help, number, asap, sent, hello, help account

Topic 4
thank, dm, sent, reply, sent dm, appreciate, response, flight, great, yes, thank help, okay, hi, ll, did

Topic 5
just, got, just got, just want, want, sent, just sent, dm, bought, know, wanted, just bought, people, ve just, just wanted



- Topic 0
time, phone, don, ve, amp, order, like, app, fix, new, going, flight, day, update, today
    - 시간 관련

- Topic 1
service, customer, customer service, worst, worst customer, terrible, care, great, poor, customer care, today, bad, experience, horrible, good
    - 불만

- Topic 2
thanks, hi, dm, reply, ok, great, ll, thanks help, thanks reply, ok thanks, response, check, sure, good, flight
    - 감사

- Topic 3
help, need, need help, account, dm, hi, email, hey, trying, thanks help, number, asap, sent, hello, help account
    - 초반에 언급될만한 단어(일반적)

- Topic 4
thank, dm, sent, reply, sent dm, appreciate, response, flight, great, yes, thank help, okay, hi, ll, did
    - 후반에 언급될만한 단어(일반적)

- Topic 5
just, got, just got, just want, want, sent, just sent, dm, bought, know, wanted, just bought, people, ve just, just wanted
    - 'just'

In [37]:
doc_topic = W / (W.sum(axis=1, keepdims=True) + 1e-10)

doc_topic_df = pd.DataFrame(
    doc_topic,
    columns = [f"Topic {i}" for i in range(n_topics)]
)

In [39]:
result_df.columns

Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id', 'clean_text',
       'word_count', 'Topic 0', 'Topic 1', 'Topic 2', 'Topic 3', 'Topic 4',
       'Topic 5'],
      dtype='object')

In [43]:
result_df = customer_sample.reset_index(drop=True).copy()
result_df = pd.concat([result_df, doc_topic_df], axis = 1)

result_df = result_df.drop(['tweet_id', 'author_id', 'inbound', 'created_at','response_tweet_id', 'in_response_to_tweet_id', 'text'], axis=1)

result_df

,clean_text,word_count,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5
0,basically for a chat to be opened from call lo...,23,0.723581,0.000000,0.000000,0.110260,0.000000,0.166159
1,ios and watchos no icon for twitter notificati...,21,0.222050,0.000000,0.000000,0.777950,0.000000,0.000000
2,hi there if i buy call of duty wwii on steam t...,22,0.572109,0.018520,0.196535,0.189838,0.022998,0.000000
3,hi why can t i pay my my dstv texts says the o...,15,0.533921,0.000000,0.180232,0.245931,0.039917,0.000000
4,trying to buy my kid nibling a keyboard for an...,23,0.632672,0.000000,0.031936,0.178702,0.009155,0.147536
...,...,...,...,...,...,...,...,...
26609,not sure this is alright lol rubyoulongtime,7,0.265625,0.000000,0.381587,0.050812,0.115524,0.186453
26610,con yo s creo en el buenfin ahora viene el bla...,21,0.897924,0.027719,0.050844,0.000000,0.000000,0.023514
26611,mobile care i need to speak to someone but i w...,19,0.335987,0.147752,0.000000,0.504626,0.011635,0.000000
26612,my iphone isn t letting me update it ain t shi...,13,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [47]:
def show_representative_tweets(df, topic_name, n=10):
    topic_docs = df.sort_values(by=topic_name, ascending=False).head(n)

    for idx, row in topic_docs.iterrows():
        print(f"[{idx}] score = {row[topic_name]:.3f}")
        print(row["clean_text"])
        print()

In [48]:
for i in range(n_topics):
    print("=" * 80)
    print(f"Representative tweets for Topic {i}")
    print("=" * 80)
    show_representative_tweets(result_df, f"Topic {i}", n=10)

Representative tweets for Topic 0
[24248] score = 1.000
that was last time for my last order should i use the same link each time

[7294] score = 1.000
fix my phone i m irritated

[19766] score = 1.000
every time i order a phone with you guys it gets lost stop using

[6951] score = 1.000
i ve been awake for less than half an hour and have used my phone for an even shorter time battery is at now fix this

[8117] score = 1.000
is this charge only because it was my first time ordering is this going to be like this all the time

[16884] score = 1.000
and i have the latest software update which is the ios so why is my phone freezing constantly with a new iphone and the latest update every time a new phone comes out are you trying to get consumers to update their plan

[25675] score = 1.000
what concerning is that each and every time i place an app order the store fucks it up and does nothing about it thats what is concerning it s like if you don t wait in line for minutes so you can babysit

In [49]:
# 본문을 봐도 일부단어에만 꽂힌 것으로 보이므로 중단

# Conclusion

The extracted topics were often dominated by generic conversational words such as “help”, “service”, and “thanks” rather than clearly separated semantic issue categories. Even representative tweets within the same topic frequently shared only superficial lexical similarities instead of broader discourse structure.

These results suggest that traditional bag-of-words topic modeling methods such as NMF may struggle to capture meaningful semantic organization in short and noisy customer-support tweets. The experiment also highlights the limitations of sparse lexical representations for modeling conversational language.